In [2]:
import numpy as np
import itertools
import math
from collections import defaultdict

def top_k_kendall_tau(a: list, b: list, k: int, p: float) -> int:
    dist = 0
    p_dist = 0
    a_dict = {a[i]: i for i in range(k)}
    b_dict = {b[i]: i for i in range(k)}
    remaining_b = set(b_dict.keys()) - set(a_dict.keys())

    for i in range(k):
        for j in range(i + 1, k):
            if a[i] not in b_dict and a[j] not in b_dict:
                p_dist += p
            else:
                if a[i] not in b_dict:
                    b_later = True
                elif a[j] not in b_dict:
                    b_later = False
                else:
                    b_later = b_dict[a[i]] > b_dict[a[j]]
                a_later = (i > j)
                dist += a_later != b_later
        for item in remaining_b:
            if a[i] not in b_dict:
                b_later = True
            else:
                b_later = b_dict[a[i]] > b_dict[item]
            a_later = False
            dist += a_later != b_later
    
    p_dist += math.comb(len(remaining_b), 2) * p
    return dist + p_dist

def mallows_prob(center: list, ref_ranking: list, k: int, p: float, beta: float):
    distance = top_k_kendall_tau(center, ref_ranking, k, p)
    return np.exp(-beta * distance)

In [3]:
k = 3
arr_a = [1, 2, 3, 4, 5]
p = 0.5123
beta_a = 0.5

top_k_sets = itertools.permutations(arr_a, k)
count = 0
total_prob = 0
sets = []
probs = []
comb_probs_dict = defaultdict(lambda: 0)

for S in top_k_sets:
    prob_num = mallows_prob(arr_a, S, k, p, beta_a)
    sets.append(S)
    probs.append(prob_num)
    total_prob += prob_num
    count += 1

print("Total:", len(sets))

for i in range(len(sets)):
    prob = probs[i] / total_prob
    print(f"Ordered Set: {sets[i]}, prob: {round(prob * 100, 3)} %")
    comb_probs_dict[frozenset(sets[i])] += prob

Total: 60
Ordered Set: (1, 2, 3), prob: 8.771 %
Ordered Set: (1, 2, 4), prob: 5.32 %
Ordered Set: (1, 2, 5), prob: 5.32 %
Ordered Set: (1, 3, 2), prob: 5.32 %
Ordered Set: (1, 3, 4), prob: 3.227 %
Ordered Set: (1, 3, 5), prob: 3.227 %
Ordered Set: (1, 4, 2), prob: 3.227 %
Ordered Set: (1, 4, 3), prob: 1.957 %
Ordered Set: (1, 4, 5), prob: 0.711 %
Ordered Set: (1, 5, 2), prob: 3.227 %
Ordered Set: (1, 5, 3), prob: 1.957 %
Ordered Set: (1, 5, 4), prob: 0.711 %
Ordered Set: (2, 1, 3), prob: 5.32 %
Ordered Set: (2, 1, 4), prob: 3.227 %
Ordered Set: (2, 1, 5), prob: 3.227 %
Ordered Set: (2, 3, 1), prob: 3.227 %
Ordered Set: (2, 3, 4), prob: 1.957 %
Ordered Set: (2, 3, 5), prob: 1.957 %
Ordered Set: (2, 4, 1), prob: 1.957 %
Ordered Set: (2, 4, 3), prob: 1.187 %
Ordered Set: (2, 4, 5), prob: 0.431 %
Ordered Set: (2, 5, 1), prob: 1.957 %
Ordered Set: (2, 5, 3), prob: 1.187 %
Ordered Set: (2, 5, 4), prob: 0.431 %
Ordered Set: (3, 1, 2), prob: 3.227 %
Ordered Set: (3, 1, 4), prob: 1.957 %
Ordere

In [4]:
print("Total:", len(comb_probs_dict))

for comb in comb_probs_dict:
    print(f"Combination: {comb}, prob: {round(comb_probs_dict[comb] * 100, 3)} %")

Total: 10
Combination: frozenset({1, 2, 3}), prob: 27.821 %
Combination: frozenset({1, 2, 4}), prob: 16.874 %
Combination: frozenset({1, 2, 5}), prob: 16.874 %
Combination: frozenset({1, 3, 4}), prob: 10.235 %
Combination: frozenset({1, 3, 5}), prob: 10.235 %
Combination: frozenset({1, 4, 5}), prob: 2.808 %
Combination: frozenset({2, 3, 4}), prob: 6.208 %
Combination: frozenset({2, 3, 5}), prob: 6.208 %
Combination: frozenset({2, 4, 5}), prob: 1.703 %
Combination: frozenset({3, 4, 5}), prob: 1.033 %


In [ ]:
def choice_probs(center: list, ref_ranking: frozenset, k: int, beta: float, null_val: int = 0):
    n = len(center)
    center_set = set(center[:k])
    S = center_set.intersection(ref_ranking)
    A_null = ref_ranking.union({null_val})
    A_bar = [a for a in ref_ranking if a not in center_set] + [null_val]
    L = A_bar + center[:k]
    
    r = len(A_bar)
    ell = len(S)
    m = len(L)

    cur_arr = [(center[i], r + i) for i in range(n) if center[i] in S]
    print("Cur Arr:", cur_arr)
    DP_table = np.zeros((m, k, ell))
    print(r, ell, m)

    for j in range(r):
        sampled_at_j_prob = 1/(n - k - j + 1)
        none_sampled_prob = 1
        for jp in range(1, j):
            none_sampled_prob *= (1 - (r / (n - k - jp)))
        DP_table[:r, j, 0] = sampled_at_j_prob * none_sampled_prob
    
    print(DP_table)

    #"""
    for q in range(1, ell):
        a_cur, cur_ind = cur_arr[q]
        if a_cur not in A_null:
            for j in range(k):
                DP_table[cur_ind, j, q] = DP_table[cur_ind, j, q-1] * PRIM_POS_SEQ(S, j, q, before=False) + DP_table[cur_ind, j-1, q-1] * PRIM_POS_SEQ(S, j-1, q, before=True)
        else:
            for j in range(k):
                DP_table[cur_ind, j, q] = DP_table[cur_ind, j, q-1] * PRIM_POS_SEQ(S, j, q, before=False)
    #"""

In [31]:
arr_h = [1, 2, 3, 4, 5]
beta_h = 0.5

ranking_a = next(iter(comb_probs_dict.keys()))
print(ranking_a)

choice_probs(arr_h, ranking_a, k, beta_h)

frozenset({1, 2, 3})
Curr Arr: [(1, 1), (2, 2), (3, 3)]
1 3 4
[[[0.33333333 0.         0.        ]
  [0.         0.         0.        ]
  [0.         0.         0.        ]]

 [[0.         0.         0.        ]
  [0.         0.         0.        ]
  [0.         0.         0.        ]]

 [[0.         0.         0.        ]
  [0.         0.         0.        ]
  [0.         0.         0.        ]]

 [[0.         0.         0.        ]
  [0.         0.         0.        ]
  [0.         0.         0.        ]]]
